# Laboratório — Amostragem, representatividade e data leakage

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/02-statistics/notebooks/12-amostragem-vies-leakage-laboratorio.ipynb)

Este laboratório usa dados **sintéticos**, pois conhecer o mecanismo gerador permite medir viés diretamente. Nenhum resultado deve ser interpretado como estimativa sobre uma população real.


## Objetivos e dependências

Vamos comparar:

1. amostra aleatória simples, amostra desproporcional e pós-estratificação;
2. viés e variabilidade em 400 repetições;
3. split aleatório e temporal sob mudança de distribuição;
4. features disponíveis e uma feature pós-desfecho;
5. split por linha e split por entidade.

Dependências: Python 3.10+, NumPy, pandas, SciPy, Matplotlib e scikit-learn. Seed fixa: `20260907`.


In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
from scipy.special import expit
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260907
CORES = {"SRS": "#0072B2", "Desproporcional": "#D55E00", "Ponderada": "#009E73"}

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## Parte A — Amostragem

### 1. Uma população finita com verdade conhecida

Criamos 100 mil unidades: 60% urbanas e 40% rurais. A probabilidade geradora de satisfação é 70% no primeiro estrato e 40% no segundo. O parâmetro de interesse será a média **realizada** da população finita.


In [ ]:
rng_pop = np.random.default_rng(SEED)
N = 100_000
segmento = np.repeat(["urbano", "rural"], [60_000, 40_000])
prob_satisfacao = np.where(segmento == "urbano", 0.70, 0.40)
satisfeito = rng_pop.binomial(1, prob_satisfacao)

populacao = pd.DataFrame({
    "id": np.arange(N),
    "segmento": segmento,
    "satisfeito": satisfeito,
})

parametro = populacao["satisfeito"].mean()
por_segmento = populacao.groupby("segmento")["satisfeito"].agg(["size", "mean"])
print(f"Parâmetro da população finita: {parametro:.6f}")
por_segmento

In [ ]:
assert len(populacao) == 100_000
assert por_segmento["size"].sum() == 100_000
assert por_segmento.loc["rural", "size"] == 40_000
assert por_segmento.loc["urbano", "size"] == 60_000
assert 0.57 < parametro < 0.59
assert populacao["id"].is_unique
print("População e schema: OK")

### 2. Três estimativas com \(n=2.000\)

- SRS: sorteio uniforme sem reposição;
- desproporcional: 1.800 urbanos e 200 rurais;
- ponderada: combina as médias dos estratos usando 60% e 40%, proporções conhecidas da população.

A ponderação corrige composição observada, mas o estrato rural continua tendo apenas 200 observações.


In [ ]:
srs = populacao.sample(2_000, random_state=SEED)
amostra_urbana = populacao.query("segmento == 'urbano'").sample(1_800, random_state=SEED + 1)
amostra_rural = populacao.query("segmento == 'rural'").sample(200, random_state=SEED + 2)
desproporcional = pd.concat([amostra_urbana, amostra_rural], ignore_index=True)

estimativa_srs = srs["satisfeito"].mean()
estimativa_ingenua = desproporcional["satisfeito"].mean()
estimativa_ponderada = (
    0.60 * amostra_urbana["satisfeito"].mean()
    + 0.40 * amostra_rural["satisfeito"].mean()
)

resultado_unico = pd.Series({
    "Parâmetro": parametro,
    "SRS": estimativa_srs,
    "Desproporcional ingênua": estimativa_ingenua,
    "Pós-estratificada": estimativa_ponderada,
})
resultado_unico.to_frame("proporção satisfeita").round(6)

In [ ]:
assert np.isclose(estimativa_srs, 0.599)
assert np.isclose(estimativa_ingenua, 0.689)
assert np.isclose(estimativa_ponderada, 0.6026666666666667)
assert abs(estimativa_ponderada - parametro) < abs(estimativa_ingenua - parametro)
print("A ponderação aproximou a estimativa do parâmetro neste sorteio.")

### 3. Viés e variabilidade em repetições

Um resultado isolado mistura erro sistemático e acaso. Repetimos os desenhos 400 vezes sem reposição em cada repetição.


In [ ]:
rng_rep = np.random.default_rng(SEED)
indices_urbanos = np.flatnonzero(segmento == "urbano")
indices_rurais = np.flatnonzero(segmento == "rural")
resultados = []

for repeticao in range(400):
    idx_srs = rng_rep.choice(N, 2_000, replace=False)
    idx_u = rng_rep.choice(indices_urbanos, 1_800, replace=False)
    idx_r = rng_rep.choice(indices_rurais, 200, replace=False)

    est_srs = satisfeito[idx_srs].mean()
    est_desproporcional = np.concatenate([satisfeito[idx_u], satisfeito[idx_r]]).mean()
    est_ponderada = 0.60 * satisfeito[idx_u].mean() + 0.40 * satisfeito[idx_r].mean()
    resultados.append((est_srs, est_desproporcional, est_ponderada))

simulacoes = pd.DataFrame(resultados, columns=["SRS", "Desproporcional", "Ponderada"])
metricas_amostragem = pd.DataFrame({
    "média_das_estimativas": simulacoes.mean(),
    "viés_empírico": simulacoes.mean() - parametro,
    "MAE": simulacoes.sub(parametro).abs().mean(),
    "RMSE": np.sqrt(simulacoes.sub(parametro).pow(2).mean()),
})
metricas_amostragem.round(6)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for coluna in simulacoes:
    ax.hist(
        simulacoes[coluna],
        bins=24,
        alpha=0.48,
        label=coluna,
        color=CORES[coluna],
    )
ax.axvline(parametro, color="black", linestyle="--", linewidth=2, label="Parâmetro")
ax.set(
    title="Distribuição de 400 estimativas",
    xlabel="Proporção estimada de satisfação",
    ylabel="Frequência",
)
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.show()

In [ ]:
assert abs(metricas_amostragem.loc["SRS", "viés_empírico"]) < 0.003
assert metricas_amostragem.loc["Desproporcional", "viés_empírico"] > 0.08
assert abs(metricas_amostragem.loc["Ponderada", "viés_empírico"]) < 0.003
assert metricas_amostragem.loc["Ponderada", "RMSE"] > metricas_amostragem.loc["SRS", "RMSE"]
print("Viés corrigido, mas com variância maior por haver apenas 200 observações rurais.")

### Leitura metodológica

A amostra desproporcional não é “ruim” por definição: estratos raros às vezes são deliberadamente superamostrados. O erro seria ignorar o desenho. A ponderação aproxima o centro correto, mas não recupera a precisão de uma amostra bem distribuída.


## Parte B — Leakage e desenho do split

### 4. Dados temporais com drift conhecido

Geramos 6.000 eventos ordenados. Após o evento 4.800, a relação entre o sinal e o alvo enfraquece e a taxa-base muda. A coluna `status_pos_desfecho` é construída a partir do alvo e só existiria depois do resultado: ela é leakage deliberado.


In [ ]:
rng_tempo = np.random.default_rng(SEED)
n_eventos = 6_000
tempo = np.arange(n_eventos)
sinal = rng_tempo.normal(size=n_eventos)
custo = rng_tempo.lognormal(mean=2.0 + 0.0001 * tempo, sigma=0.35)

coeficiente = np.where(tempo < 4_800, 1.60, 0.35)
intercepto = np.where(tempo < 4_800, -0.20, 0.50)
probabilidade = expit(intercepto + coeficiente * sinal - 0.12 * (np.log(custo) - 2))
alvo = rng_tempo.binomial(1, probabilidade)

status_pos_desfecho = np.clip(alvo + rng_tempo.normal(0, 0.08, n_eventos), 0, 1)
eventos = pd.DataFrame({
    "tempo": tempo,
    "sinal": sinal,
    "custo": custo,
    "status_pos_desfecho": status_pos_desfecho,
    "alvo": alvo,
})
eventos.head()

### 5. Split temporal treino/validação/teste

Usamos os primeiros 70% para treino, os 15% seguintes para validação e os 15% finais para teste. Nenhum registro futuro participa do ajuste.


In [ ]:
treino = eventos.iloc[:4_200].copy()
validacao = eventos.iloc[4_200:5_100].copy()
teste = eventos.iloc[5_100:].copy()

assert treino["tempo"].max() < validacao["tempo"].min()
assert validacao["tempo"].max() < teste["tempo"].min()
assert set(treino.index).isdisjoint(validacao.index)
assert set(treino.index).isdisjoint(teste.index)

pd.DataFrame({
    "n": [len(treino), len(validacao), len(teste)],
    "tempo_min": [treino.tempo.min(), validacao.tempo.min(), teste.tempo.min()],
    "tempo_max": [treino.tempo.max(), validacao.tempo.max(), teste.tempo.max()],
    "taxa_alvo": [treino.alvo.mean(), validacao.alvo.mean(), teste.alvo.mean()],
}, index=["treino", "validação", "teste"]).round(4)

### 6. Pipeline correto e feature vazada

O `SimpleImputer`, o `StandardScaler` e a regressão são ajustados somente no treino. Comparamos features disponíveis em \(t_0\) com a inclusão proibida da variável pós-desfecho.


In [ ]:
def ajustar_avaliar(colunas):
    pipeline = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1_000, random_state=SEED),
    )
    pipeline.fit(treino[colunas], treino["alvo"])
    auc_validacao = roc_auc_score(
        validacao["alvo"], pipeline.predict_proba(validacao[colunas])[:, 1]
    )
    auc_teste = roc_auc_score(
        teste["alvo"], pipeline.predict_proba(teste[colunas])[:, 1]
    )
    return pipeline, auc_validacao, auc_teste

features_disponiveis = ["sinal", "custo"]
features_com_leakage = ["sinal", "custo", "status_pos_desfecho"]

modelo_honesto, auc_val_honesta, auc_teste_honesta = ajustar_avaliar(features_disponiveis)
modelo_vazado, auc_val_vazada, auc_teste_vazada = ajustar_avaliar(features_com_leakage)

pd.DataFrame({
    "AUC validação": [auc_val_honesta, auc_val_vazada],
    "AUC teste": [auc_teste_honesta, auc_teste_vazada],
}, index=["Features disponíveis", "Com feature pós-desfecho"]).round(6)

In [ ]:
assert 0.55 < auc_teste_honesta < 0.70
assert auc_teste_honesta < auc_val_honesta
assert auc_val_vazada > 0.99 and auc_teste_vazada > 0.99
print(f"AUC honesta no futuro: {auc_teste_honesta:.6f}")
print(f"AUC impossível com leakage: {auc_teste_vazada:.6f}")

A AUC perfeita não indica um ótimo modelo: denuncia que a avaliação recebeu a resposta. O resultado honesto cai no período futuro porque houve drift.


### 7. Split aleatório pode esconder o drift

O split aleatório mistura os regimes antigo e novo. Ele responde “como o modelo vai em linhas aleatórias deste arquivo?”, não “como vai no próximo período?”.


In [ ]:
treino_aleatorio, teste_aleatorio = train_test_split(
    eventos,
    test_size=0.30,
    stratify=eventos["alvo"],
    random_state=SEED,
)
modelo_aleatorio = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1_000, random_state=SEED),
)
modelo_aleatorio.fit(treino_aleatorio[features_disponiveis], treino_aleatorio["alvo"])
auc_aleatoria = roc_auc_score(
    teste_aleatorio["alvo"],
    modelo_aleatorio.predict_proba(teste_aleatorio[features_disponiveis])[:, 1],
)

comparacao_split = pd.Series({
    "Split aleatório": auc_aleatoria,
    "Teste temporal futuro": auc_teste_honesta,
})
comparacao_split.to_frame("AUC").round(6)

In [ ]:
assert auc_aleatoria > auc_teste_honesta + 0.10
print(f"O split aleatório superestimou a AUC futura em {auc_aleatoria - auc_teste_honesta:.6f}.")

## Parte C — Leakage de entidade

### 8. Observações repetidas

Cada uma de 1.000 entidades aparece cinco vezes e possui uma assinatura estável. O rótulo é específico da entidade. Um KNN consegue “reconhecer” entidades vistas, mas não há relação generalizável para entidades novas.


In [ ]:
rng_grupo = np.random.default_rng(SEED)
n_entidades, repeticoes = 1_000, 5
entidade = np.repeat(np.arange(n_entidades), repeticoes)
assinatura_base = rng_grupo.uniform(-10, 10, n_entidades)
assinatura = np.repeat(assinatura_base, repeticoes) + rng_grupo.normal(
    0, 0.001, n_entidades * repeticoes
)
rotulo_entidade = rng_grupo.binomial(1, 0.5, n_entidades)
rotulo = np.repeat(rotulo_entidade, repeticoes)

X_grupo = assinatura.reshape(-1, 1)
y_grupo = rotulo

### 9. Split errado por linha versus split por grupo

No primeiro caso, a maioria das entidades aparece nos dois lados. No segundo, `GroupShuffleSplit` mantém entidades inteiras em apenas um lado.


In [ ]:
idx = np.arange(len(y_grupo))
idx_treino_linha, idx_teste_linha = train_test_split(
    idx, test_size=0.25, stratify=y_grupo, random_state=SEED
)
modelo_linha = KNeighborsClassifier(n_neighbors=3)
modelo_linha.fit(X_grupo[idx_treino_linha], y_grupo[idx_treino_linha])
acc_linha = accuracy_score(
    y_grupo[idx_teste_linha], modelo_linha.predict(X_grupo[idx_teste_linha])
)
sobreposicao_linha = len(
    set(entidade[idx_treino_linha]) & set(entidade[idx_teste_linha])
)

separador = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
idx_treino_grupo, idx_teste_grupo = next(
    separador.split(X_grupo, y_grupo, groups=entidade)
)
modelo_grupo = KNeighborsClassifier(n_neighbors=3)
modelo_grupo.fit(X_grupo[idx_treino_grupo], y_grupo[idx_treino_grupo])
acc_grupo = accuracy_score(
    y_grupo[idx_teste_grupo], modelo_grupo.predict(X_grupo[idx_teste_grupo])
)
sobreposicao_grupo = len(
    set(entidade[idx_treino_grupo]) & set(entidade[idx_teste_grupo])
)

pd.DataFrame({
    "acurácia": [acc_linha, acc_grupo],
    "entidades compartilhadas": [sobreposicao_linha, sobreposicao_grupo],
}, index=["Split por linha", "Split por entidade"]).round(4)

In [ ]:
assert acc_linha > 0.90
assert 0.40 < acc_grupo < 0.60
assert sobreposicao_linha > 700
assert sobreposicao_grupo == 0
print("O desempenho alto desapareceu quando a avaliação passou a usar entidades novas.")

## 10. Relatório mínimo do split

Um experimento reproduzível deve registrar contagens, intervalos temporais, prevalências, grupos e verificações de interseção. O exemplo abaixo pode ser adaptado para um pipeline real.


In [ ]:
relatorio = {
    "seed": SEED,
    "unidade": "evento",
    "unidade_de_generalizacao": "período futuro",
    "instante_de_previsao": "antes do status pós-desfecho",
    "features_permitidas": features_disponiveis,
    "features_proibidas": ["status_pos_desfecho"],
    "n_treino": len(treino),
    "n_validacao": len(validacao),
    "n_teste": len(teste),
    "ordem_temporal_valida": bool(
        treino.tempo.max() < validacao.tempo.min() < teste.tempo.min()
    ),
}
pd.Series(relatorio, name="valor")

## Desafio

1. Altere a amostra desproporcional para 50% urbano e 50% rural e repita as 400 simulações.
2. Compare viés e RMSE das três estimativas.
3. Mude o instante do drift e observe a diferença entre split aleatório e temporal.
4. Remova a feature pós-desfecho antes de qualquer seleção automática.
5. Experimente `GroupKFold` no conjunto de entidades e verifique a interseção de IDs em cada fold.

Uma resposta completa declara população, frame, probabilidades de inclusão, unidade de generalização, instante \(t_0\), regra do split e limitações.


In [ ]:
# Verificação final
assert populacao["id"].is_unique
assert abs(metricas_amostragem.loc["Desproporcional", "viés_empírico"]) > 0.08
assert abs(metricas_amostragem.loc["Ponderada", "viés_empírico"]) < 0.003
assert treino.tempo.max() < validacao.tempo.min() < teste.tempo.min()
assert auc_teste_vazada > 0.99
assert auc_aleatoria > auc_teste_honesta
assert sobreposicao_grupo == 0

print("Todas as verificações passaram.")
print(f"Parâmetro populacional: {parametro:.6f}")
print(f"Viés da amostra desproporcional: {metricas_amostragem.loc['Desproporcional', 'viés_empírico']:.6f}")
print(f"AUC aleatória: {auc_aleatoria:.6f}")
print(f"AUC temporal futura: {auc_teste_honesta:.6f}")
print(f"Acurácia por linha: {acc_linha:.4f}")
print(f"Acurácia por entidade: {acc_grupo:.4f}")

## Conclusões

- Mais observações não corrigem seleção sistemática.
- Ponderação pode corrigir composição conhecida, mas aumenta incerteza quando pesos são desiguais.
- Split aleatório e split temporal respondem perguntas diferentes.
- Uma feature pós-desfecho pode produzir métrica perfeita e totalmente inválida.
- Observações da mesma entidade precisam permanecer juntas quando o objetivo é generalizar para entidades novas.
- Pipelines protegem o pré-processamento, mas não definem a população, o instante de previsão nem o split correto.

**Fontes:** [OpenIntro IMS](https://www.openintro.org/book/ims/), [scikit-learn — data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage), [cross-validation para grupos](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data) e Kaufman et al. (2012), [DOI 10.1145/2382577.2382579](https://doi.org/10.1145/2382577.2382579).
